## **Data Processing**
### Note, run ThimkersRemoteWork.ipynb first before running this
### You must also use the exact same kernel to keep the variables
---

In [3]:
# %pip install scikit_posthocs
# %pip install scikit-learn
# %pip install plotly
# %pip install imbalanced-learn

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

from scipy import stats
from scipy.stats import mannwhitneyu

from scipy.stats import pearsonr, spearmanr, levene, f_oneway, shapiro, kruskal, chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from matplotlib.colors import ListedColormap
from imblearn.over_sampling import SMOTE

In [4]:

%store -r
print("Variables restored successfully.")


Variables restored successfully.


In [5]:
print("Current Variables")
print(f"target                      : {target.shape}")
print(f"current_profession_encoded  : {current_profession_encoded.shape}")
print(f"age_group                   : {age_group.shape}")
print(f"education_level             : {education_level.shape}")
print(f"employment_status           : {employment_status.shape}")
print(f"dev_type_encoded            : {dev_type_encoded.shape}")
print(f"work_years (non-null)       : {work_years.notna().sum()}")
print(f"learn_years (non-null)      : {learn_years.notna().sum()}")
print(f"org_size_ordinal            : {org_size_ordinal.shape}")
print(f"work_tool_count             : {work_tool_count.shape}")
print(f"personal_tool_count         : {personal_tool_count.shape}")
print(f"geographic_regions_encoded  : {geographic_regions_encoded.shape}")
print(f"language_features           : {language_features.shape}")
print(f"database_features           : {database_features.shape}")
print(f"platform_features           : {platform_features.shape}")
print(f"webframe_features           : {webframe_features.shape}")
print(f"devenv_features             : {devenv_features.shape}")
print(f"collab_features             : {collab_features.shape}")
print(f"aimodel_features            : {aimodel_features.shape}")
print(f"ai_industry_use             : {ai_industry_use.shape}")
print(f"ai_learn_how                : {ai_learn_how.shape}")
print(f"learncodeai_encoded         : {learncodeai_encoded.shape}")
print(f"aiselect_encoded            : {aiselect_encoded.shape}")
print(f"aiagents_encoded            : {aiagents_encoded.shape}")
print(f"aiagentchange_encoded       : {aiagentchange_encoded.shape}")
print(f"ai_technical_use            : {ai_technical_use.shape}")
print(f"ai_knowledge                : {ai_knowledge.shape}")
print(f"ai_orchestration            : {ai_orchestration.shape}")
print(f"ai_observe_secure           : {ai_observe_secure.shape}")
print(f"ai_external                 : {ai_external.shape}")

Current Variables
target                      : (49191,)
current_profession_encoded  : (49191, 4)
age_group                   : (49191, 6)
education_level             : (49191, 8)
employment_status           : (49191, 5)
dev_type_encoded            : (49191, 21)
work_years (non-null)       : 42893
learn_years (non-null)      : 43042
org_size_ordinal            : (49191,)
work_tool_count             : (49191,)
personal_tool_count         : (49191,)
geographic_regions_encoded  : (49191, 19)
language_features           : (49191, 42)
database_features           : (49191, 30)
platform_features           : (49191, 42)
webframe_features           : (49191, 28)
devenv_features             : (49191, 27)
collab_features             : (49191, 25)
aimodel_features            : (49191, 17)
ai_industry_use             : (49191, 10)
ai_learn_how                : (49191, 13)
learncodeai_encoded         : (49191, 2)
aiselect_encoded            : (49191, 4)
aiagents_encoded            : (49191, 4)
aiage

## **All Features and Train/Test Split**

In [6]:
X = pd.concat([
    current_profession_encoded,
    age_group,
    education_level,
    employment_status,
    dev_type_encoded,
    geographic_regions_encoded,
    pd.DataFrame({'org_size': org_size_ordinal}),
    pd.DataFrame({'work_exp': work_years}),
    pd.DataFrame({'years_code': learn_years}),
    pd.DataFrame({'work_tools': work_tool_count}),
    pd.DataFrame({'personal_tools': personal_tool_count}),
    language_features,
    database_features,
    platform_features,
    webframe_features,
    devenv_features,
    collab_features,
    aimodel_features,
    ai_industry_use,
    ai_learn_how,
    learncodeai_encoded,
    aiselect_encoded,
    aiagents_encoded,
    aiagentchange_encoded,
    ai_technical_use,
    ai_knowledge,
    ai_orchestration,
    ai_observe_secure,
    ai_external,
], axis=1)

y = target

# Remove NaN and set median data
X_clean = X.copy()
X_clean = X_clean.fillna(0)

# 
for col in ['work_exp', 'years_code', 'work_tools', 'personal_tools', 'org_size']:
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())

print(f"Full matrix: {X_clean.shape}")
print(f"Target: {y.shape}")
print(f"Target Ratio: {y.value_counts().to_dict()}")

Full matrix: (49191, 395)
Target: (49191,)
Target Ratio: {0: 34016, 1: 15175}


In [ ]:
# Three-way split: 60% train, 20% validation, 20% test
# First split: separate test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: separate train and validation from remaining 80%
# 0.25 of 80% = 20% of total for validation, leaving 60% for training
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Scale ONLY continuous features!!!
continuous_features = ['org_size', 'work_exp', 'years_code', 'work_tools', 'personal_tools']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

# Fit scaler on training data only, transform all three sets
X_train_scaled[continuous_features] = scaler.fit_transform(X_train[continuous_features])
X_val_scaled[continuous_features] = scaler.transform(X_val[continuous_features])
X_test_scaled[continuous_features] = scaler.transform(X_test[continuous_features])

# Convert to numpy arrays
X_train_scaled = X_train_scaled.values
X_val_scaled = X_val_scaled.values
X_test_scaled = X_test_scaled.values

# Oversampling with SMOTE
# Performed Worse
# from imblearn.over_sampling import SMOTE
# smote = SMOTE(random_state=42)
# X_train_scaled, y_train = smote.fit_resample(X_train_scaled, y_train)

print(f"Train size: {X_train_scaled.shape}")
print(f"Validation size: {X_val_scaled.shape}")
print(f"Test size: {X_test_scaled.shape}")
print(f"\nTrain class balance: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Validation class balance: {pd.Series(y_val).value_counts().to_dict()}")
print(f"Test class balance: {pd.Series(y_test).value_counts().to_dict()}")
print(f"\nScaled features: {continuous_features}")
print(f"One-hot encoded features remain as 0/1 (not scaled)")

Train size: (40818, 395)
Validation size: (9838, 395)
Test size: (9839, 395)

Train class balance: {0: 20409, 1: 20409}
Validation class balance: {0: 6803, 1: 3035}
Test class balance: {0: 6804, 1: 3035}

Scaled features: ['org_size', 'work_exp', 'years_code', 'work_tools', 'personal_tools']
One-hot encoded features remain as 0/1 (not scaled)


## **K-Nearest Neighbors (KNN)**

In [8]:
k_range = range(1, 31)
train_errors = []
val_errors = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_errors.append(1 - knn.score(X_train_scaled, y_train))
    val_errors.append(1 - knn.score(X_val_scaled, y_val))
    print(f"K={k:<3}  Train Error: {train_errors[-1]:.4f}  Val Error: {val_errors[-1]:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(k_range), y=train_errors,
    mode='lines+markers', name='Train Error',
    line=dict(color='royalblue')
))
fig.add_trace(go.Scatter(
    x=list(k_range), y=val_errors,
    mode='lines+markers', name='Validation Error',
    line=dict(color='orange')
))

# Get the best K based on validation error
best_k = val_errors.index(min(val_errors)) + 1
fig.add_vline(x=best_k, line_dash='dash', line_color='green',
              annotation_text=f'Best K={best_k}', annotation_position='top right')

fig.update_layout(
    title='KNN - Error Rate vs Number of Neighbors (K)',
    xaxis_title='K (n_neighbors)',
    yaxis_title='Error Rate',
    template='plotly_white',
    height=500
)
fig.show()

print(f"Best K by lowest validation error: K = {best_k}")
print(f"Train error: {train_errors[best_k - 1]:.4f}")
print(f"Validation error: {val_errors[best_k - 1]:.4f}")


K=1    Train Error: 0.0002  Val Error: 0.4130
K=2    Train Error: 0.0132  Val Error: 0.3834
K=2    Train Error: 0.0132  Val Error: 0.3834
K=3    Train Error: 0.2095  Val Error: 0.4316
K=3    Train Error: 0.2095  Val Error: 0.4316
K=4    Train Error: 0.1943  Val Error: 0.4116
K=4    Train Error: 0.1943  Val Error: 0.4116
K=5    Train Error: 0.2633  Val Error: 0.4390
K=5    Train Error: 0.2633  Val Error: 0.4390
K=6    Train Error: 0.2506  Val Error: 0.4249
K=6    Train Error: 0.2506  Val Error: 0.4249
K=7    Train Error: 0.2851  Val Error: 0.4480
K=7    Train Error: 0.2851  Val Error: 0.4480
K=8    Train Error: 0.2771  Val Error: 0.4329
K=8    Train Error: 0.2771  Val Error: 0.4329
K=9    Train Error: 0.2986  Val Error: 0.4490
K=9    Train Error: 0.2986  Val Error: 0.4490
K=10   Train Error: 0.2921  Val Error: 0.4381
K=10   Train Error: 0.2921  Val Error: 0.4381
K=11   Train Error: 0.3067  Val Error: 0.4507
K=11   Train Error: 0.3067  Val Error: 0.4507
K=12   Train Error: 0.3015  Val Er

Best K by lowest validation error: K = 2
Train error: 0.0132
Validation error: 0.3834


In [9]:
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_scaled, y_train)
knn_predictions = knn_best.predict(X_test_scaled)

cm_knn = confusion_matrix(y_test, knn_predictions)
acc_knn = accuracy_score(y_test, knn_predictions)
report_knn = classification_report(y_test, knn_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_knn, fp_knn, fn_knn, tp_knn = cm_knn.ravel()

print(f"KNN (K={best_k}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 63)
print(f"{'Accuracy':<25} {acc_knn:>12.4f}")
print(f"{'Precision':<25} {report_knn['Non-Remote']['precision']:>12.4f} {report_knn['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_knn['Non-Remote']['recall']:>12.4f} {report_knn['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_knn['Non-Remote']['f1-score']:>12.4f} {report_knn['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_knn['Non-Remote']['support']:>12} {report_knn['Remote']['support']:>12}")
print(f"{'Macro Avg F1':<25} {report_knn['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_knn['weighted avg']['f1-score']:>12.4f}")

fig = go.Figure(data=go.Heatmap(
    z=cm_knn,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_knn), str(fp_knn)], [str(fn_knn), str(tp_knn)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'KNN (K={best_k}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()


KNN (K=2) Results
Metric                      Non-Remote       Remote
---------------------------------------------------------------
Accuracy                        0.6221
Precision                       0.8098       0.4297
Recall                          0.5927       0.6880
F1 Score                        0.6845       0.5290
Support                         6804.0       3035.0
Macro Avg F1                    0.6067
Weighted Avg F1                 0.6365


## **Logistic Regression**

In [10]:
C_range = [0.001, 0.01, 0.1, 1, 10, 100]
C_strings = [str(c) for c in C_range]

# Not all penalty and solver combinations are valid
hyperparam_combos = [
    ('l2', 'lbfgs', {}),
    ('l2', 'liblinear', {}),
    ('l1', 'liblinear', {}),
    ('l1', 'saga', {}),
    ('elasticnet', 'saga', {'l1_ratio': 0.5}),
    (None, 'lbfgs', {}),
]

lr_results = {}

for i, (penalty, solver, extra) in enumerate(hyperparam_combos, 1):
    label = f"{penalty or 'none'}/{solver}"
    print(f"[{i}/{len(hyperparam_combos)}] Fitting: {label}")
    train_errors, val_errors = [], []
    for C in C_range:
        model = LogisticRegression(penalty=penalty, solver=solver, C=C, max_iter=1000, random_state=42, **extra)
        model.fit(X_train_scaled, y_train)
        train_errors.append(1 - model.score(X_train_scaled, y_train))
        val_errors.append(1 - model.score(X_val_scaled, y_val))
        print(f"  C={str(C):<8}  Train Error: {train_errors[-1]:.4f}  Validation Error: {val_errors[-1]:.4f}")
    lr_results[label] = {'train': train_errors, 'val': val_errors}

# Plot validation error for all combos
fig = go.Figure()
for combo, errors in lr_results.items():
    fig.add_trace(go.Scatter(
        x=C_strings, y=errors['val'],
        mode='lines+markers', name=combo
    ))

fig.update_layout(
    title='Logistic Regression - Validation Error vs C by Penalty/Solver',
    xaxis_title='C (Inverse Regularization Strength)',
    yaxis_title='Validation Error Rate',
    template='plotly_white',
    height=500
)
fig.show()

# Find best overall combo and C
best_lr_label, best_C, best_C_idx, best_err = None, None, None, 1.0

for combo, errors in lr_results.items():
    index = errors['val'].index(min(errors['val']))
    if errors['val'][index] < best_err:
        best_err = errors['val'][index]
        best_lr_label = combo
        best_C_idx = index
        best_C = C_range[index]

best_penalty, best_solver = best_lr_label.split('/')
best_penalty = None if best_penalty == 'none' else best_penalty
best_extra = {'l1_ratio': 0.5} if best_penalty == 'elasticnet' else {}

print(f"Best combo: {best_lr_label}")
print(f"Best C: {best_C}")
print(f"Train error: {lr_results[best_lr_label]['train'][best_C_idx]:.4f}")
print(f"Validation error: {best_err:.4f}")

[1/6] Fitting: l2/lbfgs
  C=0.001     Train Error: 0.3092  Validation Error: 0.3474
  C=0.001     Train Error: 0.3092  Validation Error: 0.3474
  C=0.01      Train Error: 0.2928  Validation Error: 0.3427
  C=0.01      Train Error: 0.2928  Validation Error: 0.3427
  C=0.1       Train Error: 0.2872  Validation Error: 0.3441
  C=0.1       Train Error: 0.2872  Validation Error: 0.3441
  C=1         Train Error: 0.2871  Validation Error: 0.3444
  C=1         Train Error: 0.2871  Validation Error: 0.3444
  C=10        Train Error: 0.2870  Validation Error: 0.3462
  C=10        Train Error: 0.2870  Validation Error: 0.3462
  C=100       Train Error: 0.2868  Validation Error: 0.3454
[2/6] Fitting: l2/liblinear
  C=100       Train Error: 0.2868  Validation Error: 0.3454
[2/6] Fitting: l2/liblinear
  C=0.001     Train Error: 0.3078  Validation Error: 0.3537
  C=0.001     Train Error: 0.3078  Validation Error: 0.3537
  C=0.01      Train Error: 0.2940  Validation Error: 0.3431
  C=0.01      Train 

c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning:

The max_iter was reached which means the coef_ did not converge



  C=1         Train Error: 0.2872  Validation Error: 0.3451


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning:

The max_iter was reached which means the coef_ did not converge



  C=10        Train Error: 0.2871  Validation Error: 0.3457


...

c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1232: UserWarning:

Setting penalty=None will ignore the C and l1_ratio parameters



  C=10        Train Error: 0.2870  Validation Error: 0.3459


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1232: UserWarning:

Setting penalty=None will ignore the C and l1_ratio parameters



  C=100       Train Error: 0.2870  Validation Error: 0.3459


Best combo: elasticnet/saga
Best C: 0.1
Train error: 0.2886
Validation error: 0.3421


In [11]:
lr_best = LogisticRegression(penalty=best_penalty, solver=best_solver, C=best_C,
    max_iter=1000, random_state=42, **best_extra
)

lr_best.fit(X_train_scaled, y_train)
lr_predictions = lr_best.predict(X_test_scaled)

cm_lr = confusion_matrix(y_test, lr_predictions)
acc_lr = accuracy_score(y_test, lr_predictions)
report_lr = classification_report(y_test, lr_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_lr, fp_lr, fn_lr, tp_lr = cm_lr.ravel()

print(f"Logistic Regression ({best_lr_label}, C={best_C}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_lr['Non-Remote']['precision']:>12.4f} {report_lr['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_lr['Non-Remote']['recall']:>12.4f} {report_lr['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_lr['Non-Remote']['f1-score']:>12.4f} {report_lr['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_lr['Non-Remote']['support']:>12} {report_lr['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_lr:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_lr['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_lr['weighted avg']['f1-score']:>12.4f}")

fig = go.Figure(data=go.Heatmap(
    z=cm_lr,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_lr), str(fp_lr)], [str(fn_lr), str(tp_lr)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'Logistic Regression ({best_lr_label}, C={best_C}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()


Logistic Regression (elasticnet/saga, C=0.1) Results
Metric                      Non-Remote       Remote
---------------------------------------------------
Precision                       0.8381       0.4643
Recall                          0.6246       0.7295
F1 Score                        0.7158       0.5675
Support                         6804.0       3035.0
---------------------------------------------------
Accuracy                        0.6570
Macro Avg F1                    0.6416
Weighted Avg F1                 0.6700


In [12]:

# Getting top variables with .coef
feature_names = X_clean.columns.tolist()
coefs = lr_best.coef_[0] 

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefs
})
coef_df['Abs'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs', ascending=False).head(20)
coef_df = coef_df.sort_values('Coefficient')

colors = ['tomato' if c < 0 else 'royalblue' for c in coef_df['Coefficient']]

fig = go.Figure(go.Bar(
    x=coef_df['Coefficient'],
    y=coef_df['Feature'],
    orientation='h',
    marker_color=colors
))
fig.update_layout(
    title=f'Logistic Regression ({best_lr_label}, C={best_C}) - Top 20 Feature Coefficients',
    xaxis_title='Coefficient Value',
    yaxis_title='Feature',
    template='plotly_white',
    height=600
)
fig.show()

print("Top 20 features by absolute coefficient:")
print(f"{'Rank':<6} {'Feature':<40} {'Coefficient':>12}")
print("-" * 60)
for rank, (index, row) in enumerate(coef_df.sort_values('Abs', ascending=False).iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<40} {row['Coefficient']:>12.4f}")


Top 20 features by absolute coefficient:
Rank   Feature                                   Coefficient
------------------------------------------------------------
1      learncodeai_no                                 1.9614
2      learncodeai_yes                                1.8778
3      employment_employed                            1.6830
4      devtype_frontend developer                     1.2154
5      devtype_mobile developer                       1.2073
6      employment_independent                         1.1602
7      devtype_backend developer                      1.0773
8      devtype_full-stack developer                   1.0226
9      devtype_game developer                         0.9677
10     devtype_qa tester                              0.9320
11     devtype_ux/ui designer                         0.9130
12     employment_student                             0.8630
13     devtype_software architect                     0.8294
14     region_eastern_asia                  

## **Support Vector Machine (SVM)**

In [13]:
#SVM Hyperparameter Tuning
C_range = [0.001, 0.01, 0.1, 1, 10, 100]
kernel_options = ['linear', 'rbf', 'poly']
svm_results = {}
for kernel in kernel_options:
    train_errors, val_errors = [], []
    print(f"Testing SVM with kernel: {kernel}")
    for C in C_range:
        svm = SVC(kernel=kernel, C=C, max_iter=10000, random_state=42)
        svm.fit(X_train_scaled, y_train)
        train_errors.append(1 - svm.score(X_train_scaled, y_train))
        val_errors.append(1 - svm.score(X_val_scaled, y_val))
        print(f"  C={str(C):<8}  Train Error: {train_errors[-1]:.4f}  Validation Error: {val_errors[-1]:.4f}")
    svm_results[kernel] = {'train': train_errors, 'val': val_errors}
# Plot SVM results
fig = go.Figure()
for kernel in kernel_options:
    fig.add_trace(go.Scatter(x=C_range, y=svm_results[kernel]['val'], mode='lines+markers', name=f'{kernel} Validation'))
    fig.add_trace(go.Scatter(x=C_range, y=svm_results[kernel]['train'], mode='lines+markers', name=f'{kernel} Train'))
fig.update_layout(
    title='SVM - Error Rate vs C by Kernel',
    xaxis_title='C (Inverse Regularization Strength)',
    yaxis_title='Error Rate',
    template='plotly_white',
    height=500
)
fig.show()
# Find best SVM combo
###
best_svm_kernel, best_svm_C, best_svm_C_idx, best_svm_err = None, None, None, 1.0
for kernel in kernel_options:
    index = svm_results[kernel]['val'].index(min(svm_results[kernel]['val']))
    if svm_results[kernel]['val'][index] < best_svm_err:
        best_svm_err = svm_results[kernel]['val'][index]
        best_svm_kernel = kernel
        best_svm_C_idx = index
        best_svm_C = C_range[index]
print(f"Best SVM combo: Kernel={best_svm_kernel}, C={best_svm_C}")
print(f"Train error: {svm_results[best_svm_kernel]['train'][best_svm_C_idx]:.4f}")
print(f"Validation error: {best_svm_err:.4f}")
svm_best = SVC(kernel=best_svm_kernel, C=best_svm_C, max_iter=10000, random_state=42)
svm_best.fit(X_train_scaled, y_train)
svm_predictions = svm_best.predict(X_test_scaled)
cm_svm = confusion_matrix(y_test, svm_predictions)
acc_svm = accuracy_score(y_test, svm_predictions)
report_svm = classification_report(y_test, svm_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)
tn_svm, fp_svm, fn_svm, tp_svm = cm_svm.ravel()
print(f"SVM (Kernel={best_svm_kernel}, C={best_svm_C}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_svm['Non-Remote']['precision']:>12.4f} {report_svm['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_svm['Non-Remote']['recall']:>12.4f} {report_svm['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_svm['Non-Remote']['f1-score']:>12.4f} {report_svm['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_svm['Non-Remote']['support']:>12} {report_svm['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_svm:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_svm['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_svm['weighted avg']['f1-score']:>12.4f}")
fig = go.Figure(data=go.Heatmap(
    z=cm_svm,
    x=['Non-Remote', 'Remote'],
    y=['Non-Remote', 'Remote'],
    colorscale='Blues',
    text=cm_svm,
    texttemplate="%{text}",
    hoverongaps=False
))
fig.update_layout(
    title='SVM Confusion Matrix',
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    template='plotly_white'
)
fig.show()

Testing SVM with kernel: linear


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning:

Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.



  C=0.001     Train Error: 0.4998  Validation Error: 0.6907


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning:

Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.



  C=0.01      Train Error: 0.5000  Validation Error: 0.6915


...

c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning:

Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.



  C=100       Train Error: 0.1168  Validation Error: 0.3665


Best SVM combo: Kernel=rbf, C=1
Train error: 0.1096
Validation error: 0.3020


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning:

Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.



SVM (Kernel=rbf, C=1) Results
Metric                      Non-Remote       Remote
---------------------------------------------------
Precision                       0.8287       0.5302
Recall                          0.7403       0.6570
F1 Score                        0.7820       0.5868
Support                         6804.0       3035.0
---------------------------------------------------
Accuracy                        0.7146
Macro Avg F1                    0.6844
Weighted Avg F1                 0.7218


## **Naive Bayes**

In [14]:
# Naive Bayes
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)
nb_predictions = nb.predict(X_test_scaled)
cm_nb = confusion_matrix(y_test, nb_predictions)
acc_nb = accuracy_score(y_test, nb_predictions)
report_nb = classification_report(y_test, nb_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)
tn_nb, fp_nb, fn_nb, tp_nb = cm_nb.ravel()
print(f"Naive Bayes Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_nb['Non-Remote']['precision']:>12.4f} {report_nb['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_nb['Non-Remote']['recall']:>12.4f} {report_nb['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_nb['Non-Remote']['f1-score']:>12.4f} {report_nb['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_nb['Non-Remote']['support']:>12} {report_nb['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_nb:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_nb['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_nb['weighted avg']['f1-score']:>12.4f}")
fig = go.Figure(data=go.Heatmap(
    z=cm_nb,
    x=['Non-Remote', 'Remote'],
    y=['Non-Remote', 'Remote'],
    colorscale='Blues',
    text=cm_nb,
    texttemplate="%{text}",
    hoverongaps=False
))
fig.update_layout(
    title="Confusion Matrix - Naive Bayes",
    xaxis_title="Predicted",
    yaxis_title="Actual"
)
fig.show()

Naive Bayes Results
Metric                      Non-Remote       Remote
---------------------------------------------------
Precision                       0.8171       0.3921
Recall                          0.4722       0.7631
F1 Score                        0.5985       0.5180
Support                         6804.0       3035.0
---------------------------------------------------
Accuracy                        0.5619
Macro Avg F1                    0.5583
Weighted Avg F1                 0.5737


## **Random Forest**

In [15]:
n_estimators_range  = [10, 50, 100, 200, 300, 500]
n_estimators_strings = [str(n) for n in n_estimators_range]

max_depth_range = [None, 5, 10, 20]
bootstrap_range = [True, False]

rf_combos = [
    (depth, option) for depth in max_depth_range for option in bootstrap_range
]

rf_results = {}

for i, (depth, boot) in enumerate(rf_combos, 1):
    label = f"depth={'None' if depth is None else depth}/bootstrap={boot}"
    print(f"[{i}/{len(rf_combos)}] Fitting: {label}")
    rf_train_errors = []
    rf_val_errors  = []
    for n in n_estimators_range:
        rf = RandomForestClassifier(n_estimators=n, max_depth=depth, bootstrap=boot, random_state=42, 
                                    n_jobs=-1)
        rf.fit(X_train_scaled, y_train)
        rf_train_errors.append(1 - rf.score(X_train_scaled, y_train))
        rf_val_errors.append(1 - rf.score(X_val_scaled, y_val))
        print(f"n_estimators={str(n):<6}  Train Error: {rf_train_errors[-1]:.4f}  Validation Error: {rf_val_errors[-1]:.4f}")
    rf_results[label] = {'train': rf_train_errors, 'val': rf_val_errors}

fig = go.Figure()
for combo, errors in rf_results.items():
    fig.add_trace(go.Scatter(
        x=n_estimators_strings, y=errors['val'],
        mode='lines+markers', name=combo
    ))

fig.update_layout(
    title='Random Forest - Validation Error vs n_estimators by max_depth/bootstrap',
    xaxis_title='n_estimators',
    yaxis_title='Validation Error Rate',
    template='plotly_white',
    height=500
)
fig.show()

best_rf_label, best_n, best_n_idx, best_rf_err = None, None, None, 1.0

for combo, errors in rf_results.items():
    idx = errors['val'].index(min(errors['val']))
    if errors['val'][idx] < best_rf_err:
        best_rf_err    = errors['val'][idx]
        best_rf_label  = combo
        best_n_idx     = idx
        best_n         = n_estimators_range[idx]

depth_choice, boot_choice = best_rf_label.split('/')
best_depth = None if 'None' in depth_choice else int(depth_choice.split('=')[1])
best_bootstrap = boot_choice.split('=')[1] == 'True'

print(f"Best combo: {best_rf_label}")
print(f"Best n_estimators: {best_n}")
print(f"Train error: {rf_results[best_rf_label]['train'][best_n_idx]:.4f}")
print(f"Validation error: {best_rf_err:.4f}")

[1/8] Fitting: depth=None/bootstrap=True
n_estimators=10      Train Error: 0.0075  Validation Error: 0.2903
n_estimators=10      Train Error: 0.0075  Validation Error: 0.2903
n_estimators=50      Train Error: 0.0002  Validation Error: 0.2739
n_estimators=50      Train Error: 0.0002  Validation Error: 0.2739
n_estimators=100     Train Error: 0.0002  Validation Error: 0.2667
n_estimators=100     Train Error: 0.0002  Validation Error: 0.2667
n_estimators=200     Train Error: 0.0002  Validation Error: 0.2592
n_estimators=200     Train Error: 0.0002  Validation Error: 0.2592
n_estimators=300     Train Error: 0.0002  Validation Error: 0.2556
n_estimators=300     Train Error: 0.0002  Validation Error: 0.2556
n_estimators=500     Train Error: 0.0002  Validation Error: 0.2552
[2/8] Fitting: depth=None/bootstrap=False
n_estimators=500     Train Error: 0.0002  Validation Error: 0.2552
[2/8] Fitting: depth=None/bootstrap=False
n_estimators=10      Train Error: 0.0002  Validation Error: 0.2861
n_es

Best combo: depth=None/bootstrap=True
Best n_estimators: 500
Train error: 0.0002
Validation error: 0.2552


In [16]:
rf_best = RandomForestClassifier(n_estimators=best_n, max_depth=best_depth,
                                 bootstrap=best_bootstrap, random_state=42, n_jobs=-1)
rf_best.fit(X_train_scaled, y_train)
rf_predictions = rf_best.predict(X_test_scaled)

cm_rf = confusion_matrix(y_test, rf_predictions)
acc_rf = accuracy_score(y_test, rf_predictions)
report_rf = classification_report(y_test, rf_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()

print(f"Random Forest ({best_rf_label}, n={best_n}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_rf['Non-Remote']['precision']:>12.4f} {report_rf['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_rf['Non-Remote']['recall']:>12.4f} {report_rf['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_rf['Non-Remote']['f1-score']:>12.4f} {report_rf['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_rf['Non-Remote']['support']:>12} {report_rf['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_rf:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_rf['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_rf['weighted avg']['f1-score']:>12.4f}")

fig = go.Figure(data=go.Heatmap(
    z=cm_rf,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_rf), str(fp_rf)], [str(fn_rf), str(tp_rf)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'Random Forest ({best_rf_label}, n={best_n}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()


Random Forest (depth=None/bootstrap=True, n=500) Results
Metric                      Non-Remote       Remote
---------------------------------------------------
Precision                       0.7882       0.6296
Recall                          0.8761       0.4722
F1 Score                        0.8298       0.5396
Support                         6804.0       3035.0
---------------------------------------------------
Accuracy                        0.7515
Macro Avg F1                    0.6847
Weighted Avg F1                 0.7403


In [17]:
# Feature Importance
feature_names = X_clean.columns.tolist()
importances = rf_best.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})
importance_df = importance_df.sort_values('Importance', ascending=False).head(20)
importance_df = importance_df.sort_values('Importance')

fig = go.Figure(go.Bar(
    x=importance_df['Importance'],
    y=importance_df['Feature'],
    orientation='h',
    marker_color='royalblue'
))
fig.update_layout(
    title=f'Random Forest ({best_rf_label}, n={best_n}) - Top 20 Feature Importances',
    xaxis_title='Importance Score',
    yaxis_title='Feature',
    template='plotly_white',
    height=600
)
fig.show()

print("Top 20 features by importance:")
print(f"{'Rank':<6} {'Feature':<40} {'Importance':>12}")
print("-" * 60)
for rank, (index, row) in enumerate(importance_df.sort_values('Importance', ascending=False).iterrows(), 1):
    print(f"{rank:<6} {row['Feature']:<40} {row['Importance']:>12.4f}")


Top 20 features by importance:
Rank   Feature                                    Importance
------------------------------------------------------------
1      org_size                                       0.1228
2      years_code                                     0.0553
3      work_exp                                       0.0469
4      employment_employed                            0.0258
5      work_tools                                     0.0181
6      personal_tools                                 0.0149
7      profession_professional dev                    0.0139
8      devtype_full-stack developer                   0.0119
9      region_northern_america                        0.0111
10     collab_jira                                    0.0106
11     devtype_backend developer                      0.0101
12     ai_learn_how_ai_codegen_tools_or_ai_enabled_apps       0.0100
13     platform_amazon_web_services_aws               0.0093
14     lang_python                            

## **Neural Network**

In [18]:

# NOTE: The full hyperparameter permutations become super slow and got stuck. 
# It didn't finish even after 9 hours. So the solver will only be Adam
# and the max iterations will be explicitly set to 200.

architectures = [
    (64,),
    (128,),
    (64, 64),
    (128, 64),
    (128, 128),
    (256, 128, 64),
]
activations = ['relu', 'tanh']
alpha_range = [0.0001, 0.001, 0.01]

mlp_combos = [
    (arch, act, alpha)
    for arch in architectures
    for act in activations
    for alpha in alpha_range
]

print(f"Total MLP combos: {len(mlp_combos)}")

mlp_results = {}

for i, (arch, act, alpha) in enumerate(mlp_combos, 1):
    label = f"{arch}/{act}/Regularization={alpha}"
    print(f"[{i}/{len(mlp_combos)}] Fitting: {label}")
    mlp = MLPClassifier(
        hidden_layer_sizes=arch,
        activation=act,
        solver='adam',
        alpha=alpha,
        max_iter=200,
        random_state=42,
        early_stopping=False
    )
    mlp.fit(X_train_scaled, y_train)
    train_err = 1 - mlp.score(X_train_scaled, y_train)
    val_err  = 1 - mlp.score(X_val_scaled, y_val)
    print(f"  Train Error: {train_err:.4f}  Validation Error: {val_err:.4f}")
    mlp_results[label] = {'train': train_err, 'val': val_err}

fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(mlp_results.keys()),
    y=[v['val'] for v in mlp_results.values()],
    name='Validation Error',
    marker_color='tomato'
))
fig.add_trace(go.Bar(
    x=list(mlp_results.keys()),
    y=[v['train'] for v in mlp_results.values()],
    name='Train Error',
    marker_color='royalblue'
))
fig.update_layout(
    title='Neural Network (MLP, Adam) - Train/Validation Error by Arch/Activation/Alpha',
    xaxis_title='Combo (Arch/Activation/Alpha)',
    yaxis_title='Error Rate',
    template='plotly_white',
    height=600,
    barmode='group',
    xaxis_tickangle=-45,
    legend=dict(font=dict(size=9))
)
fig.show()

best_mlp_label = min(mlp_results, key=lambda k: mlp_results[k]['val'])
best_mlp_err   = mlp_results[best_mlp_label]['val']

label_parts = best_mlp_label.split('/')
arch_str = label_parts[0]
best_act = label_parts[1]
best_regularizer = float(label_parts[2].replace('Regularization=', ''))
best_arch = tuple(int(x) for x in arch_str.strip('()').split(',') if x.strip())
best_solver = 'adam'

print(f"\nBest combo    : {best_mlp_label}")
print(f"Architecture  : {best_arch}")
print(f"Activation    : {best_act}")
print(f"Alpha (L2)    : {best_regularizer}")
print(f"Solver        : {best_solver}")
print(f"Train error   : {mlp_results[best_mlp_label]['train']:.4f}")
print(f"Validation error    : {best_mlp_err:.4f}")


Total MLP combos: 36
[1/36] Fitting: (64,)/relu/Regularization=0.0001


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning:

Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.



  Train Error: 0.0173  Validation Error: 0.3072
[2/36] Fitting: (64,)/relu/Regularization=0.001


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning:

Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.



  Train Error: 0.0174  Validation Error: 0.3101
[3/36] Fitting: (64,)/relu/Regularization=0.01


...

c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning:

Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.



  Train Error: 0.0035  Validation Error: 0.2986
[24/36] Fitting: (128, 64)/tanh/Regularization=0.01
  Train Error: 0.0080  Validation Error: 0.3050
[25/36] Fitting: (128, 128)/relu/Regularization=0.0001
  Train Error: 0.0080  Validation Error: 0.3050
[25/36] Fitting: (128, 128)/relu/Regularization=0.0001
  Train Error: 0.0135  Validation Error: 0.3130
[26/36] Fitting: (128, 128)/relu/Regularization=0.001
  Train Error: 0.0135  Validation Error: 0.3130
[26/36] Fitting: (128, 128)/relu/Regularization=0.001
  Train Error: 0.0066  Validation Error: 0.3037
[27/36] Fitting: (128, 128)/relu/Regularization=0.01
  Train Error: 0.0066  Validation Error: 0.3037
[27/36] Fitting: (128, 128)/relu/Regularization=0.01
  Train Error: 0.0141  Validation Error: 0.3067
[28/36] Fitting: (128, 128)/tanh/Regularization=0.0001
  Train Error: 0.0141  Validation Error: 0.3067
[28/36] Fitting: (128, 128)/tanh/Regularization=0.0001


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning:

Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.



  Train Error: 0.0023  Validation Error: 0.2987
[29/36] Fitting: (128, 128)/tanh/Regularization=0.001
  Train Error: 0.0040  Validation Error: 0.2964
[30/36] Fitting: (128, 128)/tanh/Regularization=0.01
  Train Error: 0.0040  Validation Error: 0.2964
[30/36] Fitting: (128, 128)/tanh/Regularization=0.01
  Train Error: 0.0106  Validation Error: 0.2961
[31/36] Fitting: (256, 128, 64)/relu/Regularization=0.0001
  Train Error: 0.0106  Validation Error: 0.2961
[31/36] Fitting: (256, 128, 64)/relu/Regularization=0.0001
  Train Error: 0.0046  Validation Error: 0.2912
[32/36] Fitting: (256, 128, 64)/relu/Regularization=0.001
  Train Error: 0.0046  Validation Error: 0.2912
[32/36] Fitting: (256, 128, 64)/relu/Regularization=0.001
  Train Error: 0.0047  Validation Error: 0.2957
[33/36] Fitting: (256, 128, 64)/relu/Regularization=0.01
  Train Error: 0.0047  Validation Error: 0.2957
[33/36] Fitting: (256, 128, 64)/relu/Regularization=0.01
  Train Error: 0.0084  Validation Error: 0.2955
[34/36] Fitt


Best combo    : (256, 128, 64)/tanh/Regularization=0.001
Architecture  : (256, 128, 64)
Activation    : tanh
Alpha (L2)    : 0.001
Solver        : adam
Train error   : 0.0031
Validation error    : 0.2849


In [19]:

mlp_best = MLPClassifier(
    hidden_layer_sizes=best_arch,
    activation=best_act,
    solver=best_solver,
    alpha=best_regularizer,
    max_iter=200,
    random_state=42
)
mlp_best.fit(X_train_scaled, y_train)
mlp_predictions = mlp_best.predict(X_test_scaled)

cm_mlp = confusion_matrix(y_test, mlp_predictions)
acc_mlp = accuracy_score(y_test, mlp_predictions)
report_mlp = classification_report(y_test, mlp_predictions, target_names=['Non-Remote', 'Remote'], output_dict=True)

tn_mlp, fp_mlp, fn_mlp, tp_mlp = cm_mlp.ravel()

print(f"Neural Network ({best_mlp_label}) Results")
print(f"{'Metric':<25} {'Non-Remote':>12} {'Remote':>12}")
print("-" * 51)
print(f"{'Precision':<25} {report_mlp['Non-Remote']['precision']:>12.4f} {report_mlp['Remote']['precision']:>12.4f}")
print(f"{'Recall':<25} {report_mlp['Non-Remote']['recall']:>12.4f} {report_mlp['Remote']['recall']:>12.4f}")
print(f"{'F1 Score':<25} {report_mlp['Non-Remote']['f1-score']:>12.4f} {report_mlp['Remote']['f1-score']:>12.4f}")
print(f"{'Support':<25} {report_mlp['Non-Remote']['support']:>12} {report_mlp['Remote']['support']:>12}")
print("-" * 51)
print(f"{'Accuracy':<25} {acc_mlp:>12.4f}")
print(f"{'Macro Avg F1':<25} {report_mlp['macro avg']['f1-score']:>12.4f}")
print(f"{'Weighted Avg F1':<25} {report_mlp['weighted avg']['f1-score']:>12.4f}")

fig = go.Figure(data=go.Heatmap(
    z=cm_mlp,
    x=['Predicted Non-Remote', 'Predicted Remote'],
    y=['Actual Non-Remote',    'Actual Remote'],
    text=[[str(tn_mlp), str(fp_mlp)], [str(fn_mlp), str(tp_mlp)]],
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True
))
fig.update_layout(
    title=f'Neural Network ({best_mlp_label}) - Confusion Matrix',
    template='plotly_white',
    height=450
)
fig.show()


Neural Network ((256, 128, 64)/tanh/Regularization=0.001) Results
Metric                      Non-Remote       Remote
---------------------------------------------------
Precision                       0.7924       0.5383
Recall                          0.7964       0.5321
F1 Score                        0.7944       0.5352
Support                         6804.0       3035.0
---------------------------------------------------
Accuracy                        0.7149
Macro Avg F1                    0.6648
Weighted Avg F1                 0.7144


## **Model Performance Comparison**

### Model Performance Summary Table

| Model | Non-SMOTE Hyperparameters | SMOTE Hyperparameters | Accuracy (Non-SMOTE) | Accuracy (SMOTE) |
|-------|--------------------------|----------------------|------------------------|----------------------|
| K-Nearest Neighbors | K = 29 | K = 2 | 0.7090 | 0.6221 |
| Logistic Regression | l1/liblinear, C = 0.1 | elasticnet/saga, C = 0.1 | 0.7294 | 0.6570 |
| Support Vector Machine | Kernel = rbf, C = 1 | Kernel = rbf, C = 1 | 0.7456 | 0.7146 |
| Naive Bayes | No hyperparameters | No hyperparameters | 0.6578 | 0.5619 |
| **Random Forest** | **depth=None/bootstrap=True, n_estimators = 500** | **depth=None/bootstrap=True, n_estimators = 500** | **0.7517** | **0.7515** |
| Neural Network | (256, 128, 64)/relu/Regularization=0.001 | (256, 128, 64)/tanh/Regularization=0.001 | 0.7135 | 0.7149 |

### Analysis: Impact of SMOTE on Model Performance

The results demonstrate that SMOTE is not always beneficial. In this case, the original class imbalance was manageable, and introducing synthetic entries reduced model performance rather than improving it.

---

### **Store Variables for Random Forest Cultivation**

In [ ]:
%store X_train_scaled
%store X_val_scaled
%store X_test_scaled
%store y_train
%store y_val
%store y_test
%store X_clean
%store rf_best
%store best_rf_label
%store best_n
%store best_depth
%store best_bootstrap
%store acc_rf
%store cm_rf
%store report_rf